# Análisis V3 - Modelos Paramétricos de Regresión: Lineal y Polinomial Ridge

## 1. Contexto y Objetivos
Este cuaderno documenta la evaluación de los modelos paramétricos definitivos (V3) para estimar la **Producción (qq)** en función de la **Superficie (Ha)** cultivada. El objetivo principal es corregir el severo error porcentual observado en V1 (MAPE superior al 110%) mediante una función de transferencia logarítmica y la exclusión de observaciones con apalancamiento extremo.

## 2. Metodología y Corrección Matemática
* **Datos:** Se utilizó `0901_Produccion_Arroz_V2_Regresion.xlsx` (1,373 registros), habiendo eliminado únicamente 2 valores atípicos extremos con Superficie $\ge 3,000\text{ Ha}$.
* **Transformación:** Se aplicó la transformación bilogarítmica $\ln(1 + x)$ tanto a la variable predictora como al objetivo para estabilar la varianza y linealizar la relación:
  $$X_{\text{log}} = \ln(1 + \text{Superficie}), \quad y_{\text{log}} = \ln(1 + \text{Producción})$$
* **Partición:** 75% entrenamiento y 25% evaluación (semilla aleatoria 42).
* **Modelos:**
  1. Regresión Lineal por Mínimos Cuadrados Ordinarios (`10_Regresion_Lineal_V3.py`).
  2. Regresión Polinomial de grado 2 con regularización Ridge y optimización de $\alpha$ mediante `GridSearchCV` (5-fold) (`11_Regresion_Polinomial_V3.py`).
* **Métricas:** Calculadas en quintales reales aplicando la función inversa $\text{expm1}$ sobre las predicciones logarítmicas.

## 3. Resultados y Comparación con Versiones Anteriores

| Versión | Modelo | Condición de Datos | $R^2$ Score | MAE (qq) | RMSE (qq) | MAPE (%) |
|---|---|---|:---:|:---:|:---:|:---:|
| **V1** | Lineal Simple | Crudo (con mega-outlier) | 0.7664 | 2,136.91 | 19,427.76 | 110.44% |
| **V1** | Polinomial Grado 4 | Crudo sin regularizar | 0.8193 | 1,134.59 | 3,566.07 | 69.18% |
| **V2** | Polinomial Grado 4 | Filtrado sin logaritmo | -0.1581 | 1,744.17 | 9,986.51 | 69.62% |
| **V3** | **Lineal V3** | **Filtrado + $\ln(1+x)$** | **0.6787** | 1,470.68 | **5,259.79** | **49.03%** |
| **V3** | **Polinomial Ridge V3** | **Filtrado + $\ln(1+x)$ ($\alpha=10$)** | 0.5651 | **1,365.46** | 6,119.86 | 51.05% |

### Parámetros Calibrados de la Regresión Lineal V3:
* **Intercepto ($b_0$):** $3.5059$
* **Pendiente ($b_1$):** $0.9667$
* **Fórmula Analítica de Inferencia Web:**
  $$\text{Producción (qq)} = \exp\Big(3.5059 + 0.9667 \times \ln(\text{Superficie} + 1)\Big) - 1$$

## 4. Interpretación Técnica y Agronómica
1. **Elasticidad unitaria ($b_1 = 0.9667$):** El exponente cercano a 1 en escala original indica rendimientos constantes a escala. La relación física entre tierra y cosecha es multiplicativa y homogénea en el intervalo de 1 a 2,500 hectáreas.
2. **Reducción sustancial del error relativo:** El error porcentual absoluto medio (MAPE) se redujo en más de 61 puntos porcentuales respecto a la versión V1 (descendiendo de 110.44% a 49.03%). El modelo lineal ya no sobreestima artificialmente a los pequeños productores para compensar las desviaciones del mega-productor.
3. **Pérdida de rendimiento en Polinomios:** Aunque Ridge reduce ligeramente el MAE absoluto, su $R^2$ desciende a 0.5651 debido a la penalización por curvatura en las colas de la distribución. La linealidad en espacio logarítmico demuestra mayor estabilidad general.

## 5. Visualización
![Diagnóstico Regresión Lineal V3](../03_imagenes/28_regresion_lineal_v3.png)
![Regresión Polinomial Ridge V3](../03_imagenes/29_regresion_polinomial_v3.png)

## 6. Conclusión y Veredicto de Implementación
La **Regresión Lineal V3** representa la solución paramétrica recomendada para producción. Su formulación matemática no requiere librerías pesadas en backend y puede ejecutarse directamente en el frontend web (JavaScript) con tiempos de respuesta instantáneos y un error relativo controlado por debajo del 50%.